# Getting Started – Data Engineering with Snowflake

Build an end-to-end **Ingestion → Transformation → Delivery (I-T-D)** pipeline for **Tasty Bytes**, a global food-truck company. Analysts flagged that **sales in Hamburg, Germany dropped to $0 for several days in February 2022** — this pipeline finds out why.

You'll run everything up through **Transformation** right here in this notebook. Some cells are pre-written; others you'll generate by prompting **CoCo** (the prompt is shown in the markdown cell right above each one). The **Delivery** stage (Semantic View → Cortex Agent → CoWork) is done in the Snowsight UI — see the Quickstart guide.

**Layers:** `RAW_POS` (Ingestion) → `HARMONIZED` (Transformation) → `ANALYTICS` (Delivery).

## Connect this notebook to compute

This notebook runs inside a **Git-backed Workspace**. Before running any cell:

1. Click **Connect** next to the Run button and select **Create and connect**. You can monitor the connection status in the status bar at the bottom of the notebook — wait for it to show **Connected** before proceeding.

2. Set your **role** and **warehouse** using the picker at the top-left of the Notebooks editor:
   - **Role = `ACCOUNTADMIN`**
   - **Warehouse = `COMPUTE_WH`** (the default trial warehouse)

   This is the notebook's **query warehouse** — every SQL cell and any Snowpark pushdown runs on it. (Rendering the results grid is free.) You can also set these in a cell with `USE ROLE` / `USE WAREHOUSE`.

3. Notebooks in Workspaces **don't auto-select a database or schema**, so the cells below set context explicitly (`USE DATABASE` / `USE SCHEMA`) and use fully qualified names — objects resolve wherever you run them.

4. Run cells top-to-bottom with **▶** / **Shift+Enter**, or **Run all** from the toolbar. If the session idle-suspends (~30 min of inactivity), restart it from the top-left and re-run from where you left off — created objects persist in Snowflake.

Where a markdown cell shows a **CoCo prompt**, open the **CoCo** panel, paste the prompt, compare CoCo's output against the **Expected output** in the cell, and click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

In [ ]:
-- SETUP — create the three-layer database and grant Cortex/CoWork privileges.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE tasty_bytes;
CREATE OR REPLACE SCHEMA tasty_bytes.raw_pos;
CREATE OR REPLACE SCHEMA tasty_bytes.harmonized;
CREATE OR REPLACE SCHEMA tasty_bytes.analytics;

-- Privileges needed later to create the Cortex Agent and use CoWork
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## Ingestion — install the live weather share (programmatically)

The first data source is **live weather** from Snowflake Marketplace (Pelmorex Weather Source: Frostbyte). Instead of clicking through the Marketplace UI, we install it programmatically: accept the listing's legal terms, then create a database directly from the listing. This is the repeatable, scriptable way to acquire a Marketplace dataset.

In [ ]:
-- Accept the listing terms and install the Pelmorex weather share as FROSTBYTE_WEATHERSOURCE
USE ROLE ACCOUNTADMIN;

CALL SYSTEM$ACCEPT_LEGAL_TERMS('DATA_EXCHANGE_LISTING', 'GZSOZ1LLEL');

CREATE DATABASE IF NOT EXISTS FROSTBYTE_WEATHERSOURCE
  FROM LISTING 'GZSOZ1LLEL';

## Ingestion — create the raw POS tables and ORDERS_V (boilerplate)

Run the next cell as-is. It creates the empty raw POS target tables and the `ANALYTICS.ORDERS_V` view — a denormalized flat join across the raw tables that gives every downstream query one simple source for sales. There's nothing to "solve" here.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE tasty_bytes;
USE SCHEMA raw_pos;

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.country (
    country_id       NUMBER(18,0),
    country          VARCHAR(16777216),
    iso_country      VARCHAR(16777216),
    city             VARCHAR(16777216),
    city_population  VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.franchise (
    franchise_id  NUMBER(38,0),
    first_name    VARCHAR(16777216),
    last_name     VARCHAR(16777216),
    city          VARCHAR(16777216),
    country       VARCHAR(16777216),
    e_mail        VARCHAR(16777216),
    phone_number  VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.location (
    location_id       NUMBER(19,0),
    placekey          VARCHAR(16777216),
    location          VARCHAR(16777216),
    city              VARCHAR(16777216),
    region            VARCHAR(16777216),
    iso_country_code  VARCHAR(16777216),
    country           VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.menu (
    menu_id                        NUMBER(19,0),
    menu_type_id                   NUMBER(38,0),
    menu_type                      VARCHAR(16777216),
    truck_brand_name               VARCHAR(16777216),
    menu_item_id                   NUMBER(38,0),
    menu_item_name                 VARCHAR(16777216),
    item_category                  VARCHAR(16777216),
    item_subcategory               VARCHAR(16777216),
    cost_of_goods_usd              NUMBER(38,4),
    sale_price_usd                 NUMBER(38,4),
    menu_item_health_metrics_obj   VARIANT
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck (
    truck_id             NUMBER(38,0),
    menu_type_id         NUMBER(18,0),
    primary_city         VARCHAR(16777216),
    region               VARCHAR(16777216),
    iso_region           VARCHAR(16777216),
    country              VARCHAR(16777216),
    iso_country_code     VARCHAR(16777216),
    franchise_flag       NUMBER(38,0),
    year                 NUMBER(38,0),
    make                 VARCHAR(16777216),
    model                VARCHAR(16777216),
    ev_flag              NUMBER(38,0),
    franchise_id         NUMBER(38,0),
    truck_opening_date   DATE
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_header (
    order_id                NUMBER(38,0),
    truck_id                NUMBER(38,0),
    location_id             NUMBER(19,0),
    customer_id             NUMBER(38,0),
    discount_id             VARCHAR(16777216),
    shift_id                NUMBER(38,0),
    shift_start_time        TIME(9),
    shift_end_time          TIME(9),
    order_channel           VARCHAR(16777216),
    order_ts                TIMESTAMP_NTZ(9),
    served_ts               VARCHAR(16777216),
    order_currency          VARCHAR(3),
    order_amount            NUMBER(38,4),
    order_tax_amount        VARCHAR(16777216),
    order_discount_amount   VARCHAR(16777216),
    order_total             NUMBER(38,4)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_detail (
    order_detail_id              NUMBER(38,0),
    order_id                     NUMBER(38,0),
    menu_item_id                 NUMBER(38,0),
    discount_id                  VARCHAR(16777216),
    line_number                  NUMBER(38,0),
    quantity                     NUMBER(5,0),
    unit_price                   NUMBER(38,4),
    price                        NUMBER(38,4),
    order_item_discount_amount   VARCHAR(16777216)
);

-- Denormalized flat sales view used by downstream steps and the semantic view.
CREATE OR REPLACE VIEW tasty_bytes.analytics.orders_v
AS
SELECT
    oh.order_id,
    oh.truck_id,
    oh.order_ts,
    od.order_detail_id,
    od.line_number,
    m.truck_brand_name,
    m.menu_type,
    t.primary_city,
    t.region,
    t.country,
    t.franchise_flag,
    t.franchise_id,
    f.first_name  AS franchisee_first_name,
    f.last_name   AS franchisee_last_name,
    l.location,
    oh.order_channel,
    oh.order_amount AS price,
    oh.order_total
FROM tasty_bytes.raw_pos.order_header  oh
JOIN tasty_bytes.raw_pos.order_detail   od ON oh.order_id      = od.order_id
JOIN tasty_bytes.raw_pos.truck           t  ON oh.truck_id     = t.truck_id
JOIN tasty_bytes.raw_pos.menu            m  ON od.menu_item_id = m.menu_item_id
JOIN tasty_bytes.raw_pos.franchise       f  ON t.franchise_id  = f.franchise_id
JOIN tasty_bytes.raw_pos.location        l  ON oh.location_id  = l.location_id;

### STEP 1 — Create a CSV file format

A **file format** tells Snowflake how to parse raw files during loading. We need one for the CSV data on S3.

Open the **CoCo** panel and send this prompt. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Create a CSV file format named CSV_FF in TASTY_BYTES.PUBLIC with type = 'csv'."*

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE FILE FORMAT TASTY_BYTES.PUBLIC.CSV_FF
  TYPE = 'CSV';
```
</details>

### STEP 2 — Create the external stage

A **stage** is a pointer to an external storage location (in this case, an S3 bucket) so Snowflake knows where to find the files.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Create an external stage named S3LOAD in TASTY_BYTES.PUBLIC that points to 's3://sfquickstarts/tastybytes/' and uses the CSV_FF file format."*

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE STAGE TASTY_BYTES.PUBLIC.S3LOAD
  URL = 's3://sfquickstarts/tastybytes/'
  FILE_FORMAT = TASTY_BYTES.PUBLIC.CSV_FF;
```
</details>

### STEP 3 — Load the COUNTRY table (teaching example)

`COPY INTO` is Snowflake's bulk-loading command — it reads files from a stage and inserts them into a table. This single load teaches you the pattern you'll repeat for every table.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. You should see ~30 rows load. **This single COPY INTO is the pattern for every table** — only the stage path and table name change. Optionally, copy the SQL into the notebook cell for future reference.

> *"Write a COPY INTO statement that loads data from @tasty_bytes.public.s3load/raw_pos/country/ into TASTY_BYTES.RAW_POS.COUNTRY. Use FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE) to handle extra columns in the source file."*

<details>
<summary>Expected output (check against this)</summary>

```sql
COPY INTO TASTY_BYTES.RAW_POS.COUNTRY
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/country/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);
```
</details>

### STEP 4 — Load all remaining tables (scale-up)

Now we apply the same `COPY INTO` pattern at scale — loading six more tables in one shot. CoCo will also spin up a larger warehouse for performance, then tear it down to avoid idle credit burn.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Load the remaining Tasty Bytes tables from the S3 stage @tasty_bytes.public.s3load into their corresponding tables in TASTY_BYTES.RAW_POS: FRANCHISE (raw_pos/franchise/), LOCATION (raw_pos/location/), MENU (raw_pos/menu/), TRUCK (raw_pos/truck/), ORDER_HEADER (raw_pos/order_header/), ORDER_DETAIL (raw_pos/order_detail/). Create a dedicated LARGE warehouse named LOAD_WH for the bulk load, switch to it, run the COPY INTOs, then drop LOAD_WH and switch back to COMPUTE_WH."*

> **Note:** This load covers ~1 GB of data across 6 tables and may take several minutes. Wait for all success messages before continuing.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE WAREHOUSE LOAD_WH
  WAREHOUSE_SIZE = 'LARGE'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE LOAD_WH;

COPY INTO TASTY_BYTES.RAW_POS.FRANCHISE
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/franchise/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.LOCATION
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/location/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.MENU
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/menu/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.TRUCK
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/truck/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_HEADER
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_header/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_DETAIL
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_detail/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

DROP WAREHOUSE LOAD_WH;
USE WAREHOUSE COMPUTE_WH;
```
</details>

> *"Load the remaining Tasty Bytes tables from the S3 stage @tasty_bytes.public.s3load into their corresponding tables in TASTY_BYTES.RAW_POS: FRANCHISE (raw_pos/franchise/), LOCATION (raw_pos/location/), MENU (raw_pos/menu/), TRUCK (raw_pos/truck/), ORDER_HEADER (raw_pos/order_header/), ORDER_DETAIL (raw_pos/order_detail/). Create a dedicated LARGE warehouse named LOAD_WH for the bulk load, switch to it, run the COPY INTOs, then drop LOAD_WH and switch back to COMPUTE_WH."*

> **Note:** This load covers ~1 GB of data across 6 tables and may take several minutes. Wait for all success messages before continuing.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE WAREHOUSE LOAD_WH
  WAREHOUSE_SIZE = 'LARGE'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE LOAD_WH;

COPY INTO TASTY_BYTES.RAW_POS.FRANCHISE
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/franchise/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.LOCATION
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/location/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.MENU
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/menu/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.TRUCK
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/truck/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_HEADER
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_header/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_DETAIL
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_detail/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

DROP WAREHOUSE LOAD_WH;
USE WAREHOUSE COMPUTE_WH;
```
</details>

> **Note:** This load covers ~1 GB of data across 6 tables and may take several minutes. Wait for all success messages before continuing.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE WAREHOUSE LOAD_WH
  WAREHOUSE_SIZE = 'LARGE'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE LOAD_WH;

COPY INTO TASTY_BYTES.RAW_POS.FRANCHISE
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/franchise/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.LOCATION
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/location/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.MENU
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/menu/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.TRUCK
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/truck/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_HEADER
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_header/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_DETAIL
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_detail/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

DROP WAREHOUSE LOAD_WH;
USE WAREHOUSE COMPUTE_WH;
```
</details>

## Transformation — UDFs and Dynamic Tables

Ingestion is done. Now we shape the raw data into analyst-ready tables using **SQL UDFs** and **Dynamic Tables**.

**About UDFs:** a SQL UDF **must return a value** of the type in its `RETURNS` clause — its body is a single expression whose result is returned on every call. That's what makes UDFs great for **reusable data transformations** (unit conversions, formatting, business math) you can invoke anywhere, including inside Dynamic Tables.

### STEP 1 & 2 — Create the metric-conversion UDFs

The Pelmorex data is imperial; analysts want metric. Each UDF encapsulates a single unit-conversion formula so you can call it like a built-in function anywhere in SQL.

Send each prompt to CoCo one at a time. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> **STEP 1:** *"Create a SQL UDF named FAHRENHEIT_TO_CELSIUS in TASTY_BYTES.ANALYTICS that accepts a NUMBER(35,4) parameter TEMP_F and returns the Celsius equivalent as NUMBER(35,4)."*

> **STEP 2:** *"Create a SQL UDF named INCH_TO_MILLIMETER in TASTY_BYTES.ANALYTICS that accepts a NUMBER(35,4) parameter INCH and returns the millimeter equivalent as NUMBER(35,4)."*

<details>
<summary>Expected output (check against this)</summary>

```sql
-- STEP 1
CREATE OR REPLACE FUNCTION TASTY_BYTES.ANALYTICS.FAHRENHEIT_TO_CELSIUS(TEMP_F NUMBER(35,4))
  RETURNS NUMBER(35,4)
AS $$ (TEMP_F - 32) * 5/9 $$;

-- STEP 2
CREATE OR REPLACE FUNCTION TASTY_BYTES.ANALYTICS.INCH_TO_MILLIMETER(INCH NUMBER(35,4))
  RETURNS NUMBER(35,4)
AS $$ INCH * 25.4 $$;
```
</details>

### About Dynamic Tables

A **Dynamic Table** is a table whose contents are defined by a query that Snowflake keeps up to date automatically. You write the `SELECT` once and declare a target freshness (`TARGET_LAG`); Snowflake schedules the refresh and, where possible, only reprocesses changed rows (incremental refresh). You get a view's readability with a table's query speed — no pipeline code to write or schedule.

**Not just for "non-real-time" analytics.** `TARGET_LAG` can be as low as **1 minute** (or `DOWNSTREAM`), so Dynamic Tables serve **both batch and near-real-time / low-latency analytics** — you trade freshness against cost by tuning the lag. (For the *lowest*-latency, high-concurrency serving — real-time dashboards, data APIs — Snowflake also offers **Interactive Tables**.) Docs: https://docs.snowflake.com/en/user-guide/dynamic-tables/overview

**Interoperability:** a Dynamic Table can also be a **Dynamic Iceberg Table**, storing results in open **Apache Iceberg** format on cloud storage so engines like Spark and Trino can read it directly — same refresh model. Docs: https://docs.snowflake.com/en/user-guide/dynamic-tables/create-iceberg

The four DTs build in layers: `DAILY_WEATHER_DT` → `WINDSPEED_HAMBURG_DT` and `WEATHER_HAMBURG_DT`; `SALES_HAMBURG_DT` is the sales side. `WEATHER_HAMBURG_DT` + `SALES_HAMBURG_DT` power the Semantic View.

### STEP 3 — DAILY_WEATHER_DT (base weather, full-refresh)

Joins the live `FROSTBYTE_WEATHERSOURCE` share with Hamburg postal codes.

> **Why `REFRESH_MODE = FULL`?** The source is a third-party share (a database you don't own). Snowflake can only enable the change tracking that powers *incremental* refresh on objects you own — so shares use FULL refresh (recompute each cycle).

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.DAILY_WEATHER_DT
  TARGET_LAG = '1 day'
  REFRESH_MODE = FULL
  INITIALIZE = ON_CREATE
  WAREHOUSE = COMPUTE_WH
AS
SELECT
    hd.*,
    TO_VARCHAR(hd.date_valid_std, 'YYYY-MM') AS yyyy_mm,
    pc.city_name AS city,
    'Germany' AS country_desc
FROM FROSTBYTE_WEATHERSOURCE.onpoint_id.history_day hd
JOIN FROSTBYTE_WEATHERSOURCE.onpoint_id.postal_codes pc
    ON pc.postal_code = hd.postal_code
    AND pc.country = hd.country
WHERE pc.city_name = 'Hamburg';

### STEP 4 — WINDSPEED_HAMBURG_DT

Isolates Hamburg's daily max wind speed from `DAILY_WEATHER_DT`.

> **Expected message — not an error.** Creating this table (and `WEATHER_HAMBURG_DT`) may report: *"…successfully created. FULL refresh mode was selected because: Change tracking is not supported on dynamic tables with 'FULL' REFRESH_MODE unless the Dynamic Table has FROZEN WHERE constraint specified."* It's because this DT reads from `DAILY_WEATHER_DT`, which is FULL-refresh (it reads a share). A DT built on a FULL-refresh source can't refresh incrementally either, so Snowflake selects FULL for it too. The table was created correctly — it just refreshes in full each cycle.


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.WINDSPEED_HAMBURG_DT
  TARGET_LAG = '1 day'
  WAREHOUSE = COMPUTE_WH
AS
SELECT
    country_desc,
    city,
    date_valid_std,
    MAX(max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM TASTY_BYTES.HARMONIZED.DAILY_WEATHER_DT
WHERE city = 'Hamburg'
GROUP BY country_desc, city, date_valid_std
ORDER BY date_valid_std;

### STEP 5 — WEATHER_HAMBURG_DT (metric conversions)

One row per date, with temperature and precipitation converted to metric via your UDFs. This DT powers the Semantic View.


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG_DT
  TARGET_LAG = '1 day'
  WAREHOUSE = COMPUTE_WH
AS
SELECT
    date_valid_std,
    MAX(TASTY_BYTES.ANALYTICS.FAHRENHEIT_TO_CELSIUS(avg_temperature_air_2m_f)) AS avg_temperature_celsius,
    MAX(TASTY_BYTES.ANALYTICS.INCH_TO_MILLIMETER(tot_precipitation_in)) AS avg_precipitation_mm,
    MAX(max_wind_speed_100m_mph) AS max_wind_speed_mph
FROM TASTY_BYTES.HARMONIZED.DAILY_WEATHER_DT
WHERE city = 'Hamburg'
GROUP BY date_valid_std;

### STEP 6 — SALES_HAMBURG_DT (Hamburg sales with a date spine)

The sales-side counterpart to `WEATHER_HAMBURG_DT`. It filters sales to Hamburg and adds a **date spine** — a generated row for every calendar day — so days with **zero orders still appear**. Without the spine, gap days would be missing entirely and the Semantic View couldn't detect the sales drop.

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.SALES_HAMBURG_DT
  TARGET_LAG = '1 day'
  WAREHOUSE = COMPUTE_WH
AS
WITH date_spine AS (
    SELECT DATEADD(DAY, SEQ4(), '2019-01-01') AS order_date
    FROM TABLE(GENERATOR(ROWCOUNT => 3000))
)
SELECT
    ds.order_date,
    ZEROIFNULL(SUM(o.price)) AS daily_sales,
    COUNT(o.order_id) AS num_orders
FROM date_spine ds
LEFT JOIN TASTY_BYTES.ANALYTICS.ORDERS_V o
    ON DATE(o.order_ts) = ds.order_date
    AND o.country = 'Germany'
    AND o.primary_city = 'Hamburg'
GROUP BY ds.order_date
ORDER BY ds.order_date;

## Transformation complete → on to Delivery (in the UI)

You now have four Dynamic Tables that stay fresh automatically:

- `DAILY_WEATHER_DT` — Hamburg postal-code weather, full-refresh
- `WINDSPEED_HAMBURG_DT` — Hamburg wind speed over time
- `WEATHER_HAMBURG_DT` — Hamburg weather in metric units (one row/day)
- `SALES_HAMBURG_DT` — Hamburg sales by day, including zero-sales days

**Delivery** is done in the Snowsight UI (no code): build a **Semantic View** over `SALES_HAMBURG_DT` + `WEATHER_HAMBURG_DT`, create a **Cortex Agent** backed by it, and ask your question in **Snowflake CoWork**. Follow the **"Deliver With Cortex Agent and CoWork"** section of the Quickstart guide.

## Teardown

Once you're done with the lab, run the cell below to remove all objects and stop ongoing credit consumption. The Dynamic Tables in this pipeline refresh automatically — they will continue consuming credits until dropped.

> **Note:** Dropping `TASTY_BYTES` cascades to everything inside it (schemas, tables, dynamic tables, UDFs, semantic view, cortex agent). You don't need to drop individual objects.

In [ ]:
-- TEARDOWN — remove all lab objects and stop credit consumption
USE ROLE ACCOUNTADMIN;

-- Drops all schemas, tables, dynamic tables, UDFs, semantic view, and cortex agent
DROP DATABASE IF EXISTS TASTY_BYTES;

-- Removes the Marketplace weather share
DROP DATABASE IF EXISTS FROSTBYTE_WEATHERSOURCE;

-- Removes the Git API integration created for the workspace
DROP API INTEGRATION IF EXISTS GITHUB_SFC_GH_KENGUYEN;